<a href="https://colab.research.google.com/github/sunainakhatwani12/flyrank-ml-internship-sunaina/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunainakhatwani12/flyrank-ml-internship-sunaina/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define the selected lane and ML task.

lane = "Refresh / Content Opportunity Scoring"
task_type = "Ranking / scoring"
unit_of_analysis = "one pseudonymized content page"
actor = "content editor or SEO reviewer"
output = "ranked page-review queue"

print("Lane:", lane)
print("Task type:", task_type)
print("Unit of analysis:", unit_of_analysis)
print("Actor:", actor)
print("Output:", output)

Lane: Refresh / Content Opportunity Scoring
Task type: Ranking / scoring
Unit of analysis: one pseudonymized content page
Actor: content editor or SEO reviewer
Output: ranked page-review queue


## 1. My lane as an ML task (type)

My chosen lane is **Lane 2: Refresh / Content Opportunity Scoring**.

I will frame this primarily as a **ranking/scoring task**. The system should assign each content page a priority score and rank the pages from highest to lowest review priority.

This is more useful than producing only a yes/no classification because a content editor has limited time and needs to know **which pages should be reviewed first**. The highest-ranked pages would be inspected for possible refresh, expansion, protection, pruning, or monitoring.

The output is decision support for a content editor or SEO reviewer. It does not automatically decide that a page must be changed.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Load the starter dataset and sketch a temporary target column.

from pathlib import Path
import pandas as pd

possible_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"),
]

data_path = next((path for path in possible_paths if path.exists()), None)

if data_path is not None:
    df = pd.read_csv(data_path)
    source_used = str(data_path)
else:
    raw_url = (
        "https://raw.githubusercontent.com/"
        "flyrank-bih/flyrank-ml-internship-starter/main/"
        "data/raw/content_refresh_anonymized.csv"
    )
    df = pd.read_csv(raw_url)
    source_used = raw_url

required_columns = {
    "content_id",
    "client_id",
    "trend_direction",
    "impressions_90d",
    "content_age_days",
}

missing_columns = required_columns.difference(df.columns)

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {sorted(missing_columns)}"
    )

# Temporary proxy only—not a final future-outcome target.
df["future_decline_proxy"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

print("Loaded from:", source_used)
print("Dataset shape:", df.shape)
print("\nTemporary target distribution:")
print(df["future_decline_proxy"].value_counts(dropna=False).sort_index())

Loaded from: https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv
Dataset shape: (30000, 45)

Temporary target distribution:
future_decline_proxy
0    13738
1    16262
Name: count, dtype: int64


## 2. Target or proxy

The ideal target would be an **observed future outcome** showing whether a page experienced a meaningful decline after the feature-measurement period.

For example, using the larger time-based warehouse dataset, the target could be:

**future_decline = 1** when a page's search clicks or impressions decrease meaningfully during a later outcome window, compared with an earlier reference window.

Otherwise:

**future_decline = 0**

This would be an observed target because it would be calculated from performance that happened after the input features were measured.

For this starter notebook, I will sketch the target using `trend_direction == "down"` as a temporary proxy. However, this is not my final target because `trend_direction` is calculated from `trend_pct`. Therefore, `trend_direction`, `trend_pct`, and `is_declining_label` must not be used as model features. Using them as features would create target leakage and would only teach the model to reproduce an existing rule.

The temporary proxy is only being used to demonstrate the expected shape of the target column.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define and demonstrate the primary evaluation metric.

K = 50
provisional_precision_target = 0.60

proxy_base_rate = df["future_decline_proxy"].mean()
minimum_relevant_pages = int(K * provisional_precision_target)

print("Primary metric: Precision@50")
print(f"Temporary proxy base rate: {proxy_base_rate:.3f}")
print(f"Provisional success threshold: {provisional_precision_target:.2f}")
print(
    f"At least {minimum_relevant_pages} of the top {K} pages "
    "should be relevant to meet this provisional threshold."
)

Primary metric: Precision@50
Temporary proxy base rate: 0.542
Provisional success threshold: 0.60
At least 30 of the top 50 pages should be relevant to meet this provisional threshold.


## 3. Success metric

My primary success metric will be **Precision@50**.

Precision@50 measures how many of the first 50 pages in the ranked review queue actually experience the observed future outcome.

For example, if 35 of the top 50 recommended pages later show a meaningful decline, the Precision@50 would be:

**35 ÷ 50 = 0.70, or 70%**

I chose this metric because editor time is limited. The quality of the pages at the top of the queue matters more than overall accuracy across all pages.

A useful result should:

1. Perform better than a transparent fixed-rule baseline.
2. Produce a Precision@50 meaningfully above the target's overall base rate.
3. Ideally achieve at least **0.60 Precision@50**, meaning at least 30 of the first 50 recommendations are relevant.

This threshold is provisional and may be refined after the true future target and baseline performance are measured.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create the lane's working slice.
# One row represents one pseudonymized content page.

lane_columns = [
    "content_id",
    "client_id",
    "content_type",
    "main_intent",
    "content_age_days",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "future_decline_proxy",
]

available_columns = [
    column for column in lane_columns if column in df.columns
]

lane_df = (
    df.loc[
        (df["impressions_90d"].fillna(0) > 0)
        & (df["content_age_days"].fillna(0) >= 90),
        available_columns,
    ]
    .drop_duplicates(subset="content_id")
    .copy()
)

print("Unit of analysis: one pseudonymized content page")
print("Lane slice shape:", lane_df.shape)
print("Unique content pages:", lane_df["content_id"].nunique())
print("Unique clients:", lane_df["client_id"].nunique())

lane_df.head(10)

Unit of analysis: one pseudonymized content page
Lane slice shape: (30000, 12)
Unique content pages: 30000
Unique clients: 32


,content_id,client_id,content_type,main_intent,content_age_days,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,engagement_rate,future_decline_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,187,3803,29,17,0.76,10.6,5.88,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,445,15320,7,9,0.05,20.3,0.00,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,141,12581,11,11,0.09,36.5,0.00,1
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,463,11751,58,78,0.49,6.2,1.28,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,263,19140,24,145,0.13,44.0,0.00,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,transactional,147,3970,1,5,0.03,8.5,0.00,1
6,content_9a34b442b552,client_8722616204,keyword article,informational,90,20,0,1,0.00,7.0,0.00,1
7,content_a63219c6e95a,client_19581e27de,keyword article,commercial,445,1724,1,28,0.06,21.2,3.57,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,informational,90,32574,29,68,0.09,46.0,5.88,1
9,content_c27558df2b0c,client_19581e27de,keyword article,informational,257,1240,2,3,0.16,4.9,0.00,1


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one pseudonymized content page**.

Each row represents one content item belonging to one pseudonymized client. The columns describe characteristics and trailing performance signals for that page, such as its content type, age, impressions, clicks, sessions, average search position, CTR, and engagement rate.

For this lane, I will initially focus on pages that:

* Have received at least one search impression.
* Are at least 90 days old.
* Have enough history to be considered for review.

The target column shown below is the temporary `future_decline_proxy`. In the final project, it should be replaced by a target calculated from a later, non-overlapping outcome window.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build a simple fixed-rule baseline and inspect whether one threshold
# behaves consistently across different content types.

rule_df = lane_df.copy()

valid_ctr = rule_df["ctr"].dropna()
ctr_threshold = valid_ctr.median() if not valid_ctr.empty else 0

rule_df["fixed_rule_candidate"] = (
    (rule_df["impressions_90d"].fillna(0) >= 500)
    & (rule_df["content_age_days"].fillna(0) >= 180)
    & (rule_df["avg_position"].fillna(0).between(1, 20))
    & (rule_df["ctr"].fillna(ctr_threshold) <= ctr_threshold)
)

fixed_rule_pages = rule_df[rule_df["fixed_rule_candidate"]]

if len(fixed_rule_pages) > 0:
    fixed_rule_precision = fixed_rule_pages[
        "future_decline_proxy"
    ].mean()
else:
    fixed_rule_precision = float("nan")

print("Fixed CTR threshold:", round(float(ctr_threshold), 3))
print("Pages selected by fixed rule:", len(fixed_rule_pages))
print(
    "Temporary proxy precision of fixed rule:",
    round(float(fixed_rule_precision), 3)
    if pd.notna(fixed_rule_precision)
    else "No pages selected",
)

# Show that outcome rates can differ by content type.
content_type_summary = (
    rule_df.groupby("content_type", dropna=False)
    .agg(
        pages=("content_id", "nunique"),
        proxy_decline_rate=("future_decline_proxy", "mean"),
        median_impressions=("impressions_90d", "median"),
        median_ctr=("ctr", "median"),
    )
    .sort_values("pages", ascending=False)
)

content_type_summary.head(10)

Fixed CTR threshold: 0.07
Pages selected by fixed rule: 1353
Temporary proxy precision of fixed rule: 0.669


,pages,proxy_decline_rate,median_impressions,median_ctr
content_type,,,,
keyword article,27207,0.560959,955.0,0.09
feedly article,2096,0.286737,4.0,0.00
comparison article,697,0.572453,107.0,0.00


## 5. Why ML beats a fixed rule here

A transparent fixed rule is an important baseline, but one rule is unlikely to rank every page correctly.

A page's review priority may depend on interacting signals such as:

* How many impressions and clicks it receives.
* Its average search position.
* Its CTR relative to other pages in a similar position.
* Its age and content type.
* Its engagement and session activity.
* Differences between clients and content categories.
* Missing data patterns.

For example, low CTR is not equally concerning for every search position. A CTR value that is weak for a page ranking near the top of page one may be normal for a page ranking much lower. Similarly, an old page with low traffic may not deserve attention, while an old page with strong demand and weakening performance may be important.

A fixed rule would require manually selecting thresholds and could treat different clients and content types unfairly. A learned scoring method can combine several signals, learn interactions, and produce a smoother priority ranking.

However, ML only earns its place if it beats a simple and explainable baseline on held-out data. I will therefore compare the learned ranking with a fixed-rule baseline rather than assuming that ML is automatically better.

The final output will remain **directional decision support**. An editor will still inspect the page and choose the appropriate content action.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.